In [ ]:
import numpy as np
import tensorflow.keras as keras


data = np.load('Data.npz')
lst = data.files
train_x, train_y = data['train_x'], data['train_y']
validate_x, validate_y = data['valid_x'], data['valid_y']

def extract_class(label):
    s = str(label).strip()
    return s[32:]

train_y_clean    = np.array([extract_class(c) for c in train_y])
validate_y_clean = np.array([extract_class(c) for c in validate_y])

all_classes = np.unique(np.concatenate([train_y_clean, validate_y_clean]))
classes_to_index = {c: i for i, c in enumerate(all_classes)}
index_to_classes = {i: c for i, c in enumerate(all_classes)}

NUM_CLASSES = len(all_classes)

train_x_norm = np.array([x / (np.max(np.abs(x)) + 1e-8) for x in train_x])
validate_x_norm = np.array([x / (np.max(np.abs(x)) + 1e-8) for x in validate_x])

train_x_small = train_x_norm[:, ::10, :]
validate_x_small = validate_x_norm[:, ::10, :]

y_train_enc = np.array([classes_to_index[c] for c in train_y_clean], dtype=np.int32)

idx = np.random.permutation(len(train_x_small))
train_x_small = train_x_small[idx]
y_train_enc_s = y_train_enc[idx]

model = keras.Sequential([
    keras.Input(shape=(8000, 1)),
    keras.layers.Conv1D(64,  64, activation='relu', padding='same', strides=2),
    keras.layers.Conv1D(128, 32, activation='relu', padding='same', strides=2),
    keras.layers.Conv1D(256, 16, activation='relu', padding='same', strides=2),
    keras.layers.Conv1D(256,  8, activation='relu', padding='same', strides=2),
    keras.layers.GlobalAveragePooling1D(),
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dropout(0.5),
    keras.layers.Dense(NUM_CLASSES, activation='softmax'),
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    train_x_small, y_train_enc_s,
    epochs=100,
    batch_size=32,
    validation_data=(validate_x_small, y_valid_enc),
    callbacks=[keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=15, restore_best_weights=True
    )],
    verbose=1
)

loss, acc = model.evaluate(validate_x_small, y_valid_enc, verbose=0)
print(f'Val accuracy: {acc:.4f}')

if acc > 0.07:
    model.save('model.h5')
    print('Модель сохранена!')

In [1]:
import numpy as np
import tensorflow.keras as keras

I0000 00:00:1773571991.918509   62946 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1773571992.766996   62946 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1773571998.314781   62946 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
